<a href="https://colab.research.google.com/github/daniel64bit/distribuicao-renovavel/blob/dev/notebooks/entregas/entrega_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  <img src="https://raw.githubusercontent.com/scalabrinig/cdProjetoAplicadoIV/d093146488f56dfcf0ef286bcee8efe0e71b9c76/figuras/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2024/02**


# **Entrega 3**

---

# **Título do Projeto**
---

In [ ]:
#@title **Identificação do Grupo e Opção do Projeto**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
Aluno1 = 'Aluno 1, 123456789' #@param {type:"string"}
Aluno2 = 'Aluno 2, 123456789' #@param {type:"string"}
Aluno3 = 'Aluno 3, 123456789' #@param {type:"string"}
Aluno4 = 'Aluno 4, 123456789' #@param {type:"string"}
Aluno5 = 'Aluno 5, 123456789' #@param {type:"string"}





# **Introdução**

Juntar os tópicos de contexto, motivação, objetivo geral, objetivos específicos (incluir a bases de dados que será explorada) e justificativa em um único tópico.

# **Referencial Teórico**

(1 página) Defir resumidamente os principais conceitos envolvidos na sua solução. Discutir os trabalhos correlacionados. Apresentar alternativas de solução que foram empregadas no mesmo problema ou problemas semelhantes, e suas vantagens e limitações. Indicar referências ao longo do texto (padrão ABNT).



# **Diagrama de Solução**

Apresente e discuta o diagrama da solução proposta. Utilize o pipeline proposto inicialmente como base.

# **EDA e Pré-processamento dos dados**

Exploração e análise dos dados. Discussão e análise dos dados empregados (qualidade, limitações, simplicações ou recortes adotados etc.). Tarefas de preparação dos dados (transformações, compactação e encodes, junções de dados etc.).

## 1. **ANEEL**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Importação dos dados

df_dadoscomerciais = pd.read_csv('../../data/external/ANEEL/dados_comerciais/indger-dados-comerciais.csv', sep=";", encoding='latin1')
df_municipios = pd.read_excel('../../data/external/IBGE/RELATORIO_DTB_BRASIL_MUNICIPIO.xlsx')

FileNotFoundError: [Errno 2] No such file or directory: '../../data/external/ANEEL/dados_comerciais/indger-dados-comerciais.csv'

In [ ]:
# Análise inicial da base de dados da ANEEL

df_dadoscomerciais.head()

In [ ]:
pd.set_option('display.max_rows', None)

print(f"""Tipo de dados:
{df_dadoscomerciais.dtypes}

Colunas:
{df_dadoscomerciais.columns}

Formato:
{df_dadoscomerciais.shape[0]} Linhas e {df_dadoscomerciais.shape[0]} Colunas

Nulos:
{df_dadoscomerciais.isnull().sum()}
""")

In [ ]:
# Corrigindo os tipos de dado.

pd.reset_option('display.max_rows')


string_cols = ['SigAgente', 'NomAgente', 'NomTipoOutorga']
float_cols = ['VlrRestAnte', 'VlrRestAtrasado', 'VlrRestPendente', 'VlrRestPendenteAtrasado','VlrPendentePgtRessarcDanoDefe',
              'VlrPagoRessarcDano', 'MdaTempoMedAtendimentoPosto', 'MdaTempoMedAtendPostoDiaFort', 'VlrTotCompSuspIndevida']
dt_cols = ['DatGeracaoConjuntoDados', 'DatReferenciaInformada', 'DthCarga']

df_dadoscomerciais[float_cols] = df_dadoscomerciais[float_cols].replace({",":"."}, regex=True)

df_dadoscomerciais[string_cols] = df_dadoscomerciais[string_cols].astype(str)
df_dadoscomerciais[float_cols] = df_dadoscomerciais[float_cols].astype(float)
df_dadoscomerciais[dt_cols] = df_dadoscomerciais[dt_cols].apply(pd.to_datetime)

In [ ]:
# Análise inicial da base de dados do IBGE.


df_municipios.head()

In [ ]:
municipios_cols = ['UF', 'Nome_UF', 'Município', 'Código Município Completo', 'Nome_Município']
df_municipios_filter = df_municipios[municipios_cols]
print(f"""Tipo de dados:
{df_municipios_filter.dtypes}

Colunas:
{df_municipios_filter.columns}

Formato:
{df_municipios_filter.shape[0]} Linhas e {df_municipios_filter.shape[0]} Colunas

Nulos:
{df_municipios_filter.isnull().sum()}

""")

In [ ]:
# Corrigindo os tipos dos dados

municipios_str_cols = ['Nome_UF', 'Nome_Município']
df_municipios_filter.loc[:, municipios_str_cols] = df_municipios_filter[municipios_str_cols].astype(str)

In [ ]:
#Realizando o cruzamento das tabelas para obter as informações dos municípios.

df_dadoscomerciais_merge = df_dadoscomerciais.merge(df_municipios_filter,
                                                    how = 'left',
                                                    left_on = 'CodMunicipioIBGE',
                                                    right_on ='Código Município Completo',
                                                    )

df_dadoscomerciais_merge

In [ ]:
# Aqui estamos realizando um filtro para manter somente as colunas relevantes para nossa análise.

df_dadoscomerciais_merge_final = df_dadoscomerciais_merge[['NumCNPJ',
                                                            'SigAgente',
                                                            'NomAgente',
                                                            'DatReferenciaInformada',
                                                            'CodMunicipioIBGE',
                                                            'UF',
                                                            'Nome_UF',
                                                            'Nome_Município',
                                                            ]]
df_dadoscomerciais_merge_final

In [ ]:
# Realizando uma agregação para por UF e Concessionária para uma sumarização.

df_concessionarias_por_uf = df_dadoscomerciais_merge_final.groupby(['Nome_UF', 'UF', 'SigAgente', 'NumCNPJ']).agg(UltimoRegistro=('DatReferenciaInformada', 'max')).reset_index()
df_concessionarias_por_uf.head()


In [ ]:
# Realizando uma transformação para validar a última data de atuação de cada concessionária em cada estado.
# Adicionando uma nova coluna que contém a contagem do número de concessionárias por estado.

df_concessionarias_por_uf["DataMax"] = df_concessionarias_por_uf.groupby("SigAgente")["UltimoRegistro"].transform("max")
df_concessionarias_por_uf = df_concessionarias_por_uf[
    df_concessionarias_por_uf["UltimoRegistro"] == df_concessionarias_por_uf["DataMax"]
]
df_concessionarias_por_uf["Quantidade_Concessionárias_UF"] = df_concessionarias_por_uf.groupby("Nome_UF")["SigAgente"].transform("count")


In [ ]:
# Removendo a coluna redundante.

df_concessionarias_por_uf = df_concessionarias_por_uf.drop("DataMax", axis=1)
df_concessionarias_por_uf.head()

In [ ]:
# Realizando uma agregação para por Município e Concessionária para uma sumarização.


df_concessionarias_por_municipio = df_dadoscomerciais_merge_final.groupby(['Nome_Município', 'SigAgente', 'NumCNPJ']).agg(UltimoRegistro=('DatReferenciaInformada', 'max')).reset_index()


In [ ]:
# Realizando uma transformação para validação sobre a última data de atuação da Concessionária em cada município.


df_concessionarias_por_municipio["DataMax"] = df_concessionarias_por_municipio.groupby("SigAgente")["UltimoRegistro"].transform("max")
df_concessionarias_por_municipio = df_concessionarias_por_municipio[
    df_concessionarias_por_municipio["UltimoRegistro"] == df_concessionarias_por_municipio["DataMax"]
]
df_concessionarias_por_municipio["Quantidade_Concessionárias_Município"] = df_concessionarias_por_municipio.groupby("Nome_Município")["SigAgente"].transform("count")



In [ ]:
# Removendo a coluna redundante.

df_concessionarias_por_municipio = df_concessionarias_por_municipio.drop("DataMax", axis=1)
df_concessionarias_por_municipio.head()

In [ ]:
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.barplot(data=df_concessionarias_por_uf,
              x="Nome_UF",
              y="Quantidade_Concessionárias_UF",
              color="Darkblue",
              order = df_concessionarias_por_uf['Nome_UF'].value_counts().index

)
plt.title("Quantidade de Concessionárias por UF", fontsize = 16)
plt.xlabel("Estado")
plt.ylabel("Quantidade de Concessionárias")
plt.xticks(rotation=90);
plt.ylim(0,30)


In [ ]:
# Quantidade de Concessionárias por estado em forma de tabela.

sumarizacao_uf = (df_concessionarias_por_uf[["Nome_UF", "Quantidade_Concessionárias_UF"]]
        .drop_duplicates().
        sort_values(by="Quantidade_Concessionárias_UF", ascending=False)
        .reset_index(drop=True))
sumarizacao_uf

In [ ]:
# Dados estatísticos sobre as informações obtidas.

sumarizacao_uf.describe()


In [ ]:
df_concessionarias_por_uf.to_csv('../../data/processed/ANEEL/df_concessionarias_por_uf.csv')
df_concessionarias_por_municipio.to_csv('../../data/processed/ANEEL/df_concessionarias_por_municipio.csv')

## 1. **ONS**

# **Modelo base**

Aplicação de, pelo menos, um primeiro modelo (modelo de aprendizado de máquina ou modelo estatístico) que deve ser refinado até a entrega final do projeto. Apresente o método aplicado e a análise dos resultados obtidos.

In [ ]:
# Códigos aqui

# **Cronograma**

Atualizar o cronograma, se necessário, das atividades.

# **Referências**

Complementar a entrega anterior.




In [ ]:
#@title **Avaliação**
EDA_e_preprocessamento = 10 #@param {type:"slider", min:0, max:10, step:1}

Modelo_base = 10 #@param {type:"slider", min:0, max:10, step:1}

Revisao = 10 #@param {type:"slider", min:0, max:10, step:1}

Apresentacao_geral = 10 #@param {type:"slider", min:0, max:10, step:1}








In [ ]:
#@title **Nota Final**
nota = 0.4*EDA_e_preprocessamento + 0.2*Modelo_base + 0.2*Revisao + 0.2*Apresentacao_geral

print(f'Nota final do trabalho {nota :.1f}')

import numpy as np
import pandas as pd

alunos = pd.DataFrame()

lista_nome = []

for i in range(1,6):
  exec("if Aluno" + str(i) + " !='None':  lista = Aluno" + str(i) + ".split(','); lista_nome.append(lista[0]);")

alunos['nome'] = lista_nome
alunos['nota'] = np.round(nota,1)
print()
display(alunos)

Nota final do trabalho 10.0



,nome,nota
0,Aluno 1,10.0
1,Aluno 2,10.0
2,Aluno 3,10.0
3,Aluno 4,10.0
4,Aluno 5,10.0
